In [ ]:
import os
import re
from pathlib import Path

def convert_yaml_content(content: str) -> str:
    """
    Finds multi-line single-quoted 'question:' blocks and converts them
    to YAML literal block scalar (|) syntax.
    """
    # Regex to capture:
    # Group 1: Leading indentation before question:
    # Group 2: Content on the first line after question: '
    # Group 3: Subsequent multi-line content up to the final quote line
    pattern = r"^(\s*)question:\s*'((?:''|[^'\n])*)\n((?:''|[^'])*?)\n\s*'"

    def replacer(match):
        indent = match.group(1)      # e.g., "  " or ""
        first_line = match.group(2)  # First line text
        rest_lines = match.group(3)  # Rest of the multiline text

        # Replace escaped single quotes ('') with regular single quotes (')
        first_line_clean = first_line.replace("''", "'")
        rest_lines_clean = rest_lines.replace("''", "'")

        # Indent line 1 by 2 extra spaces relative to question:
        return f"{indent}question: |\n{indent}  {first_line_clean}\n{rest_lines_clean}"

    return re.sub(pattern, replacer, content, flags=re.MULTILINE)


def batch_convert_folder(input_folder: str, output_folder: str):
    input_path = Path(input_folder)
    output_path = Path(output_folder)
    
    # Create output directory if it doesn't exist
    output_path.mkdir(parents=True, exist_ok=True)

    # Find all .yaml and .yml files
    yaml_files = list(input_path.glob("*.yaml")) + list(input_path.glob("*.yml"))

    if not yaml_files:
        print(f"No YAML files found in: {input_folder}")
        return

    print(f"Found {len(yaml_files)} YAML file(s). Processing...\n")

    for file_path in yaml_files:
        # Read file with UTF-8 encoding
        content = file_path.read_text(encoding="utf-8")

        # Perform conversion
        converted_content = convert_yaml_content(content)

        # Write converted content to output directory
        out_file = output_path / file_path.name
        out_file.write_text(converted_content, encoding="utf-8")
        print(f"✓ Converted: {file_path.name} -> {out_file}")

    print("\nAll files successfully processed!")


if __name__ == "__main__":
    # Specify your folder paths here:
    INPUT_FOLDER = "./input_yamls"
    OUTPUT_FOLDER = "./output_yamls"

    batch_convert_folder(INPUT_FOLDER, OUTPUT_FOLDER)